# Experimento Geral — Previsão de Despesas Liquidadas

**Fonte:** `ts_liquidado.parquet`  
**Modelos**: ARIMA · LR (Ridge) · SVR · MLP  
**Seleção**: Menor RMSE no holdout  
**Previsão**: Julho e Agosto de 2025  

---

## Metodologia Resumida

| Etapa | Decisão |
|---|---|
| Filtragem | Apenas séries com último período = Jun/2025 |
| Pré-processamento | Mês 13 filtrado (ajustes contábeis) |
| Sinal negativo | Shift: `+|min|+1` (reversível) |
| Sinal misto | Sem transformação |
| Holdout | `min(12, 20% da série)`, mínimo 3 meses |
| Lags (SVR/MLP) | `min(12, 30% do treino)`, mínimo 3 |
| Avaliação holdout | One-step-ahead rolling forecast |
| Previsão final | Multi-step-ahead (modelo retreinado em treino+teste) |
| SVR/MLP | Desabilitados se treino < 15 obs. |

Ver `README.md` para a metodologia completa.

> **Pré-requisito**: copie `ts_liquidado.parquet` para a pasta `data/` antes de executar.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
import pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

## 1. Executar o Experimento

In [ ]:
from experiment import rodar_experimento

resultados = rodar_experimento(
    salvar_resultados=True,
    gerar_graficos=True,
    verbose=True,
)

## 2. Métricas Comparativas

In [ ]:
from reports.plots import plotar_resumo_modelos

metricas_df = resultados['metricas_df']
plotar_resumo_modelos(metricas_df)

In [ ]:
# Tabela completa de métricas
(
    metricas_df
    .sort_values(['Codigo', 'RMSE'])
    .style
    .apply(lambda row: ['background-color: #EAF7EF' if row['Selecionado']
                        else '' for _ in row], axis=1)
    .format({'RMSE': '{:,.2f}', 'MAE': '{:,.2f}', 'MAPE': '{:.2f}%'})
    .hide(axis='index')
)

## 3. Parâmetros Selecionados

In [ ]:
params_df = resultados['params_df']
display(params_df.style.hide(axis='index'))

## 4. Previsões Finais — Jul/Ago 2025

In [ ]:
prev_df = resultados['prev_df']

cols_display = [
    'Codigo', 'Nome', 'Grupo_Sinal', 'Melhor_Modelo',
    'RMSE_Teste', 'MAPE_Teste_Pct',
    'Ultimo_Real_fmt', 'Prev_Jul_2025_fmt', 'Prev_Ago_2025_fmt',
    'Var_Jul_vs_Jun_Pct', 'Var_Ago_vs_Jul_Pct',
]

display(
    prev_df[cols_display]
    .style
    .format({
        'RMSE_Teste'        : '{:,.2f}',
        'MAPE_Teste_Pct'    : '{:.2f}%',
        'Var_Jul_vs_Jun_Pct': '{:+.2f}%',
        'Var_Ago_vs_Jul_Pct': '{:+.2f}%',
    })
    .applymap(
        lambda v: 'color: green' if isinstance(v, (int, float)) and v > 0
                  else ('color: red' if isinstance(v, (int, float)) and v < 0 else ''),
        subset=['Var_Jul_vs_Jun_Pct', 'Var_Ago_vs_Jul_Pct']
    )
    .hide(axis='index')
)

## 5. Distribuição dos Modelos Vencedores

In [ ]:
import matplotlib.pyplot as plt

vencedores = prev_df['Melhor_Modelo'].value_counts()
cores = {'ARIMA': '#C0392B', 'SVR': '#F39C12', 'MLP': '#8E44AD', 'LR': '#E67E22'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    vencedores.index, vencedores.values,
    color=[cores.get(m, '#2E6DA4') for m in vencedores.index],
    edgecolor='white', width=0.5
)
for i, (idx, val) in enumerate(vencedores.items()):
    axes[0].text(i, val + 0.05, str(val), ha='center', fontweight='bold')
axes[0].set_title('Modelos Vencedores — Contagem', fontweight='bold')

por_sinal = prev_df.groupby(['Grupo_Sinal', 'Melhor_Modelo']).size().unstack(fill_value=0)
por_sinal.plot(kind='bar', ax=axes[1],
               color=[cores.get(c, '#2E6DA4') for c in por_sinal.columns],
               edgecolor='white')
axes[1].set_title('Modelos Vencedores por Grupo de Sinal', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(title='Modelo')

plt.tight_layout()
plt.show()